## 05.01节练习参考答案

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU

---

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) ，并在此基础上补充了 PyPTO 的实现。  

### 练习5.1.1
如果将 MySequential 中存储块的方式更改为 Python 列表，会出现什么样的问题？

**解答：**
如果将 MySequential 中存储块的方式从 OrderedDict 更改为 Python 列表,代码可以正常计算。但无法像 _modules 一样使用 `net.state_dict()` 方便的访问模型的网络结构和参数。

In [3]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

class MySequential_list(nn.Module):
    # 使用list
    def __init__(self, *args):
        super(MySequential_list, self).__init__()
        self.sequential = []
        for module in args:
            self.sequential.append(module)

    def forward(self, X):
        for module in self.sequential:
            X = module(X)
        return X


X = torch.rand(1,10)
net = MySequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 10))
net_list = MySequential_list(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 10))
# 结果一样
print(net(X))
print(net_list(X))
# 使用_modules方便打印net的网络结构和参数，而list则无法做到
print(net, '\n', net.state_dict())
print(net_list, '\n', net_list.state_dict())

tensor([[ 0.0243, -0.2523,  0.2086,  0.3082, -0.1241, -0.0221,  0.0659, -0.1263,
          0.0456, -0.2995]], grad_fn=<AddmmBackward0>)
tensor([[-0.1189,  0.5044,  0.3602,  0.0375, -0.2494,  0.1655,  0.1930, -0.2877,
          0.0648, -0.0686]], grad_fn=<AddmmBackward0>)
MySequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=10, bias=True)
) 
 OrderedDict([('0.weight', tensor([[-0.1819, -0.1603,  0.2741,  0.0283, -0.1550, -0.2119, -0.2957,  0.0590,
         -0.2807,  0.2843],
        [ 0.0617, -0.0506,  0.1290, -0.0999, -0.0818, -0.1985, -0.2616,  0.1858,
          0.1481, -0.1441],
        [-0.0401,  0.1100, -0.2804,  0.2580,  0.0637,  0.0406, -0.0998,  0.0387,
          0.0360,  0.2165],
        [ 0.0846, -0.2227, -0.2820,  0.0350, -0.1734, -0.2930,  0.2736, -0.2756,
          0.1203,  0.1864],
        [ 0.0068,  0.0889,  0.0143,  0.1851, -0.2636, -0.2867,  0.3022, -0.1475,
         -0.0589, -0.0594],
        [

**PyPTO 版**

In [6]:
# PyPTO 版：权重在 npu:0 上，所以输入 X 也需放到 npu 上
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

class MySequential_list(nn.Module):
    def __init__(self, *args):
        super(MySequential_list, self).__init__()
        self.sequential = []
        for module in args:
            self.sequential.append(module)

    def forward(self, X):
        for module in self.sequential:
            X = module(X)
        return X

X = torch.rand(1, 10).npu()
net = MySequential(PyPTOLinear(10, 20), PyPTOReLU(), PyPTOLinear(20, 10))
net_list = MySequential_list(PyPTOLinear(10, 20), PyPTOReLU(), PyPTOLinear(20, 10))

print(net(X))
print(net_list(X))
print(net, '\n', net.state_dict())
print(net_list, '\n', net_list.state_dict())

tensor([[-0.1729, -0.1885, -0.1285, -0.2235, -0.0672, -0.1538,  0.0560,  0.1076,
         -0.0344, -0.0034]], device='npu:0', grad_fn=<ViewBackward0>)
tensor([[ 0.5327, -0.1568, -0.2830,  0.2009, -0.1798,  0.0035, -0.3089, -0.1125,
         -0.3430,  0.5133]], device='npu:0', grad_fn=<ViewBackward0>)
MySequential(
  (0): PyPTOLinear()
  (1): PyPTOReLU()
  (2): PyPTOLinear()
) 
 OrderedDict([('0.weight', tensor([[-2.5361e-01,  1.2971e-01, -1.1083e-01, -1.2660e-01, -2.8266e-01,
         -1.0793e-02,  2.5121e-01, -1.2464e-01, -2.6586e-02, -1.8852e-01],
        [ 2.8106e-01,  1.5328e-01, -2.1924e-01, -7.2127e-02, -1.2405e-01,
         -2.9356e-02,  3.0570e-01,  2.7571e-01,  3.8010e-02, -2.2988e-01],
        [-2.9571e-01, -4.4236e-02, -1.4328e-01,  1.6155e-01, -5.5171e-02,
          1.2827e-01, -6.3909e-02, -3.6897e-02, -5.3005e-02,  1.9985e-01],
        [-1.3512e-01,  4.7149e-02,  1.2690e-02, -4.8064e-02,  1.3611e-01,
          9.7316e-02, -1.8943e-01, -4.9340e-03, -5.6222e-02,  8.5626e-02

### 练习5.1.2
实现一个块，它以两个块为参数，例如net1和net2，并返回前向传播中两个网络的串联输出。这也被称为平行块。

**解答：**
在本书 7.4 节中 GoogLeNet 模型中的 Inception 块使用了平行块技术。 下面代码实现了一个并行网络，由两个子网络组成。输入数据先分别经过两个子网络的计算，分别得到两个部分的输出结果，然后在通道维度上合并结果得到最终输出。  

其中，net1 和 net2 分别表示两个子网络，`torch.cat` 表示在指定维度上拼接张量。输出结果的大小为 `torch.Size([2, 36])`，其中第一个维度表示 batch_size 为 2，第二个维度表示输出特征图的通道数为 36（12+24）。

In [7]:
class Parallel(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1=net1 # 第一个子网络
        self.net2=net2 # 第二个子网络
        
    def forward(self,X):
        x1= self.net1(X) # 第一个子网络的输出
        x2= self.net2(X) # 第二个子网络的输出
        return torch.cat((x1,x2),dim=1) # 在通道维度上合并输出结果
      
X = torch.rand(2,10) # 输入数据
net = Parallel(nn.Sequential(nn.Linear(10,12),nn.ReLU()), nn.Sequential(nn.Linear(10,24),nn.ReLU())) # 实例化并行网络
output = net(X)
output.size() # 输出结果的大小

torch.Size([2, 36])

**PyPTO 版**

In [9]:
class Parallel(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2

    def forward(self, X):
        x1 = self.net1(X)
        x2 = self.net2(X)
        return torch.cat((x1, x2), dim=1)

X = torch.rand(2,10).npu()
net = Parallel(nn.Sequential(PyPTOLinear(10,12),PyPTOReLU()), nn.Sequential(PyPTOLinear(10,24),PyPTOReLU())) # 实例化并行网络
output = net(X)
output.size()

torch.Size([2, 36])

---

### 练习5.1.3
假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

**解答：**

In [2]:
def create_network(num_instances, input_size, hidden_size, output_size):
    # 工厂函数：每次调用都返回一个全新的、参数独立的块
    def make_block():
        return nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, input_size)
        )

    # 每次循环都调用工厂函数，产生 num_instances 个独立实例
    instances = [make_block() for _ in range(num_instances)]
    network = nn.Sequential(*instances)

    # 添加输出层
    output_layer = nn.Linear(input_size, output_size)
    network.add_module("output", output_layer)

    return network

# 示例用法
net = create_network(num_instances=3, input_size=10, hidden_size=5, output_size=2)
print(net)

# 验证各块参数独立：取第 0、1 个块的第一个参数做对比
# p0 is p1 为 True 表示共享同一份权重（错误写法）；为 False 表示各自独立（正确写法）
p0 = list(net[0].parameters())[0]
p1 = list(net[1].parameters())[0]
print("两个块是否共用同一份权重（True=共享, False=独立）:", p0 is p1)

Sequential(
  (0): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=10, bias=True)
  )
  (1): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=10, bias=True)
  )
  (2): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=10, bias=True)
  )
  (output): Linear(in_features=10, out_features=2, bias=True)
)
两个块是否共用同一份权重（True=共享, False=独立）: False


参考的答案的写法为：

In [ ]:
linear_layer = nn.Sequential(...)
instances = [linear_layer for _ in range(num_instances)]

列表推导式不会拷贝对象，instances 中的元素指向同一个 `nn.Sequential` 实例，即 `instances[0] is instances[1]` 为 True。传入 `nn.Sequential(*instances)` 后，三个槽位注册的是同一对象，参数完全共享，反向传播时梯度叠加到同一组权重上，与“多个实例”的题意相悖。

**PyPTO 版：**

In [12]:
def create_network(num_instances, input_size, hidden_size, output_size):
    def make_block():
        return nn.Sequential(
            PyPTOLinear(input_size, hidden_size), PyPTOReLU(),
            PyPTOLinear(hidden_size, input_size)
        )

    instances = [make_block() for _ in range(num_instances)]
    network = nn.Sequential(*instances)

    output_layer = PyPTOLinear(input_size, output_size)
    network.add_module("output", output_layer)

    return network

net = create_network(num_instances=3, input_size=10, hidden_size=5, output_size=2)
print(net)

p0 = list(net[0].parameters())[0]
p1 = list(net[1].parameters())[0]
print("两个块是否共用同一份权重（True=共享, False=独立）:", p0 is p1)

Sequential(
  (0): Sequential(
    (0): PyPTOLinear()
    (1): PyPTOReLU()
    (2): PyPTOLinear()
  )
  (1): Sequential(
    (0): PyPTOLinear()
    (1): PyPTOReLU()
    (2): PyPTOLinear()
  )
  (2): Sequential(
    (0): PyPTOLinear()
    (1): PyPTOReLU()
    (2): PyPTOLinear()
  )
  (output): PyPTOLinear()
)
两个块是否共用同一份权重（True=共享, False=独立）: False
